In [2]:
from pyspark.sql import SparkSession

In [3]:
spark=SparkSession.builder.appName("Case-study-question").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 13:30:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df1=spark.read.csv("data/customers.csv",header=True)

26/06/16 13:30:29 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
df1.first()

Row(customer_id='1', customer_name='Customer_1', city='Columbus', state='OH', registration_date='2023-10-17', customer_segment='VIP')

In [6]:
df1.createOrReplaceTempView("customers")

In [7]:
products=spark.read.csv("data/products.csv",header=True)
orders=spark.read.csv("data/orders.csv",header=True)
order_items=spark.read.csv("data/order_items.csv",header=True)
returns=spark.read.csv("data/returns.csv",header=True)


In [8]:
products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
returns.createOrReplaceTempView("returns")

In [9]:
customers_count=spark.sql(''' select count(customer_id) as total_count from customers''')
customers_count.show()
products_count=spark.sql(''' select count(product_id) as total_count from products''')
products_count.show()
returns_count=spark.sql(''' select count(return_id) as total_count from returns''')
returns_count.show()
orders_count=spark.sql(''' select count(order_id) as total_count from orders''')
orders_count.show()


+-----------+
|total_count|
+-----------+
|     100000|
+-----------+

+-----------+
|total_count|
+-----------+
|      50000|
+-----------+

+-----------+
|total_count|
+-----------+
|     100000|
+-----------+



[Stage 15:=============================>                            (1 + 1) / 2]

+-----------+
|total_count|
+-----------+
|    1000000|
+-----------+



In [10]:
customers_count.write.mode("overwrite").csv("Output/q1/Cust_count",header=True)
products_count.write.mode("overwrite").csv("Output/q1/Prod_count",header=True)
returns_count.write.mode("overwrite").csv("Output/q1/Retu_count",header=True)
orders_count.write.mode("overwrite").csv("Output/q1/Order_count",header=True)

In [11]:
print("total_customers:" , df1.count())
print("total_products:" , products.count())
print("total_returns:" , returns.count())
print("total_orders:" , orders.count())


total_customers: 100000
total_products: 50000
total_returns: 100000
total_orders: 1000000


In [15]:
products.show(1)
df1.show(1)
orders.show(1)
order_items.show(1)
returns.show(1)

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
+----------+------------+--------------+-------+---------+
only showing top 1 row

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 1 row

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|payment_mode|order_status|
+--------+-----------+----------+------------+------------+
|       1|      54630|2024-01-25| Credit Card|   Delivered|
+--------+-----------+-----

In [17]:
#Find total sales amount generated by each product category
sale_cat=spark.sql('''
select p.category , sum(oi.selling_price * oi.quantity) as Total from 
products p inner join order_items oi on p.product_id= oi.product_id group by p.category ''')
sale_cat.show()

[Stage 50:=============================>                            (1 + 1) / 2]

+--------------+-------------------+
|      category|              Total|
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|        Sports|7.433388681300008E8|
|   Electronics|7.442665041099958E8|
|      Clothing|7.419227945699946E8|
|         Books|7.464907783499908E8|
|        Beauty|7.626693058999963E8|
|          Toys|7.446190722999846E8|
+--------------+-------------------+



In [19]:
sale_cat.write.mode("overwrite").csv("Output/q2",header=True)

In [20]:
# top 10 customers on basis of sales amount 
total_customer=spark.sql('''
select o.customer_id , sum(oi.selling_price * oi.quantity) as Total_purchases from orders o inner join order_items oi on o.order_id = oi.order_id
group by o.customer_id  order by Total_purchases desc ''')
total_customer.show(10)

[Stage 61:=============================>                            (1 + 1) / 2]

+-----------+------------------+
|customer_id|   Total_purchases|
+-----------+------------------+
|      93094|181569.68000000005|
|      64560|169060.39999999997|
|      23289|          161573.8|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|151037.32000000004|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|         148281.04|
+-----------+------------------+
only showing top 10 rows



In [21]:
total_customer.write.mode("overwrite").csv("Output/q3",header=True)

In [22]:
#montly sales trend of latest year
month_trend=spark.sql('''select month(o.order_date) as Months ,sum(oi.selling_price * oi.quantity) as TOTAL_SALES from orders o
inner join order_items oi on o.order_id=oi.order_id where year(o.order_date)
=(select max(year(order_date)) from orders) Group by Months order by Months ''')
month_trend.show()

[Stage 91:=============================>                            (1 + 1) / 2]

+------+--------------------+
|Months|         TOTAL_SALES|
+------+--------------------+
|     1| 4.445777757600014E8|
|     2|4.1536614419999766E8|
|     3| 4.436282454099968E8|
|     4|4.2782097433999556E8|
|     5|4.4481061894999766E8|
|     6| 4.317051540600035E8|
|     7| 4.436705191200028E8|
|     8| 4.410951770200006E8|
|     9|4.3107152608000004E8|
|    10| 4.413637893100021E8|
|    11|4.3362336404000014E8|
|    12| 4.427129083499984E8|
+------+--------------------+



In [23]:
month_trend.write.mode("overwrite").csv("Output/q4",header=True)

In [25]:
# find the return percentage for each product category 
return_per = spark.sql('''
select p.category ,count(distinct r.return_id) /count(distinct oi.order_id) *100.0 as Returnpercentage  from products p  join  order_items oi on 
p.product_id=oi.product_id left join returns r on oi.order_id=r.order_id group by p.category 
''')
return_per.show()

[Stage 121:>                                                        (0 + 2) / 2]

+--------------+------------------+
|      category|  Returnpercentage|
+--------------+------------------+
|Home & Kitchen|10.028612547426862|
|        Sports|10.030642037406412|
|   Electronics|10.019234484807065|
|      Clothing| 9.971104498210204|
|         Books|10.023020257826888|
|        Beauty|10.020229033806487|
|          Toys| 10.04393793766305|
+--------------+------------------+



In [26]:
return_per.write.mode("overwrite").csv("Output/q5",header=True)

In [27]:
# determine the prefered payment mode in each state 
pay_sta=spark.sql('''
 with temp as (select c.state , o.payment_mode , count(o.payment_mode) as Total from customers c inner join orders o on c.customer_id=o.customer_id 
group by c.state , o.payment_mode),
rnk as(select * , rank() over(partition by state order by Total desc ) as Rank from temp)
 select state,payment_mode , Total from rnk where Rank=1''')
pay_sta.show()

[Stage 134:============================>                            (1 + 1) / 2]

+-----+----------------+-----+
|state|    payment_mode|Total|
+-----+----------------+-----+
|   CA|             UPI|20246|
|   FL|      Debit Card|20010|
|   GA|     Net Banking|20041|
|   IL|Cash on Delivery|20498|
|   MI|      Debit Card|20416|
|   NC|     Net Banking|19596|
|   NY|      Debit Card|20369|
|   OH|     Net Banking|20351|
|   TX|             UPI|20065|
|   WA|             UPI|20244|
+-----+----------------+-----+



In [28]:
pay_sta.write.mode("overwrite").csv("Output/q6",header=True)

In [29]:
# Identify customers who have perchased products from atleast five different category and spend more than 1 lakh 
cust_pro=spark.sql('''
select c.customer_id, count(distinct p.category) as Total , round(sum(oi.selling_price*oi.quantity),2) as Spend 
from customers c inner join orders o on c.customer_id = o.customer_id inner join order_items oi on o.order_id=oi.order_id
inner join products p on oi.product_id=p.product_id group by c.customer_id having count(distinct p.category)>=5 AND Spend>100000 order by Spend desc''')
cust_pro.show()

26/06/16 13:46:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 13:46:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 157:>                                                        (0 + 2) / 2]

+-----------+-----+---------+
|customer_id|Total|    Spend|
+-----------+-----+---------+
|      93094|    7|181569.68|
|      64560|    7| 169060.4|
|      23289|    7| 161573.8|
|      52275|    7|153364.79|
|      61218|    7|153067.55|
|      52034|    7|152680.05|
|      40442|    7|151037.32|
|      60528|    7|148691.95|
|      84830|    7|148363.84|
|      82593|    7|148281.04|
|      20648|    7|148084.29|
|      15759|    7|147852.55|
|      36102|    7| 147833.9|
|      60538|    7|147514.06|
|      79352|    7| 146656.2|
|      67896|    7|146604.06|
|      89114|    7|146574.77|
|      17810|    7|145903.82|
|      28584|    7|145728.93|
|      82699|    7| 144907.5|
+-----------+-----+---------+
only showing top 20 rows



In [30]:
cust_pro.write.mode("overwrite").csv("Output/q7",header=True)

26/06/16 13:47:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 13:47:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [31]:
# find top 3 products by revenue with each category 
rev_cat=spark.sql('''
with temp as (select p.category, p.product_name , sum(oi.quantity * oi.selling_price) as REVENUE from products p inner join order_items 
oi on p.product_id = oi.product_id
group by p.category,p.product_name),
rnk as(select * , rank() over(partition by category order by REVENUE desc) as rak from temp)
select * from rnk where rak<=3''')
rev_cat.show()

[Stage 191:============================>                            (1 + 1) / 2]

+--------------+-------------+------------------+---+
|      category| product_name|           REVENUE|rak|
+--------------+-------------+------------------+---+
|        Beauty|Product_44016|         277567.99|  1|
|        Beauty|Product_14849|274894.19999999995|  2|
|        Beauty|  Product_786|272174.69999999995|  3|
|         Books|Product_35314|         296468.78|  1|
|         Books|Product_28311|286757.72000000003|  2|
|         Books|Product_37479|276736.70999999996|  3|
|      Clothing| Product_7025| 293821.9699999999|  1|
|      Clothing| Product_1560| 288474.0900000001|  2|
|      Clothing|Product_31322| 282241.1699999999|  3|
|   Electronics| Product_6719|         299113.87|  1|
|   Electronics|Product_23519|         289561.72|  2|
|   Electronics|Product_38170|         288875.23|  3|
|Home & Kitchen| Product_5012| 305836.2200000001|  1|
|Home & Kitchen|Product_37452| 286817.4199999999|  2|
|Home & Kitchen|Product_27682|283340.36000000004|  3|
|        Sports|Product_4170

In [32]:
rev_cat.write.mode("overwrite").csv("Output/q8",header=True)